# Lab 08-03 — Communities + map/reduce summarization (the GraphRAG index step 2)

**Track 08 · GraphRAG** — turning the entity graph from Lab 01 into the *index* GraphRAG actually queries.

Lab 01 produced an entity graph; this lab turns it into the index GraphRAG actually queries. The recipe, drawn inline:

`passages (rag-mini-wikipedia parquet) → ChatOllama json_object triple extraction → networkx entity graph → Leiden communities (scikit-network; networkx Louvain fallback) → map: per-community summaries (ChatOllama) → reduce: one global corpus summary (ChatOllama)`

This notebook is **self-contained**: it imports `langchain-ollama`, `pandas`, `networkx`, and `scikit-network` directly — no repo component library. Every block of the pipeline is built right here: the local-LLM wrapper, the triple extractor, the graph builder, the community detector, and both summarization prompts all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

Community detection partitions the graph with the **Leiden** algorithm (networkx **Louvain** fallback) so densely connected entities land in the same community — unsupervised structure discovery that makes GraphRAG useful over a whole corpus without a query in sight. The **map** step summarizes every community's internal relations into a few prose sentences; the **reduce** step folds those into one global corpus summary. Detection runs fully locally (no model call); only the two summarization steps talk to the LLM, so the whole index costs roughly one LLM call per passage plus one per community.


## Setup

Two prerequisites must hold before this notebook will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM behind both the triple extraction and the map/reduce summaries (talked to through `langchain-ollama`). Fully local: no API key, no quota.
- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet`, already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-ollama`, `pandas`, `networkx`, `scikit-network`, `numpy`, and `scipy`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no sys.path trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   langchain-ollama  -> ChatOllama, the local Ollama chat backend
#   pandas            -> read the rag-mini-wikipedia parquet
#   networkx          -> the entity graph + Louvain fallback
#   scikit-network    -> the Leiden community detector (pulls numpy/scipy)
%pip install -q langchain-ollama pandas networkx scikit-network


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

# pandas + networkx + langchain-ollama — the only libraries this notebook
# needs at import time. Nothing is imported from the repo's src/ component
# library (numpy/scipy/sknetwork are pulled lazily inside the community
# detector, exactly like the shared tool does).
import networkx as nx
import pandas as pd
from langchain_ollama import ChatOllama

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 20` caps the corpus at a deterministic head — each passage costs one entity-extraction LLM call, so the pool size *is* the extraction budget. `MAX_COMMUNITIES = 6` is the map/reduce cap: the largest communities get summarized, bounding the map step at six LLM calls no matter how many communities Leiden finds. `SEED = 42` makes the community detection deterministic (Leiden's `random_state` and the Louvain fallback's `seed`), so the partition is reproducible across runs.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 20  # deterministic head; each passage costs one extraction call
MAX_COMMUNITIES = 6  # map/reduce cap: largest communities get summarized
SEED = 42  # deterministic community detection (Leiden / Louvain)


## 2. Load — first N passages of rag-mini-wikipedia

The corpus is `rag-mini-wikipedia` — the same parquet Lab 01 used — and we take the first `N_PASSAGES` passages by file order. The head is deterministic, which matters twice: the extracted graph, and therefore the communities and every downstream summary, is identical across runs.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — build graph, detect communities, map/reduce summaries

The index recipe, in three timed stages, all built inline:

- **Build** — the same inline pieces as Lab 01: `_OllamaLLM` (ChatOllama with code-fence stripping and JSON retries), `extract_triples` (one LLM call per passage), and `build_entity_graph` (networkx fold). This is the only stage that touches every passage.
- **Detect** — `detect_communities` partitions the graph with sknetwork's Leiden (networkx Louvain fallback). Fully local, no model call. Isolated nodes, which Leiden drops from its membership matrix, are recovered as singleton communities so the partition always covers the whole node set.
- **Map / reduce** — `community_summaries` renders each community's intra-community edges as relation lines (`community_text`) and asks the LLM to condense them into 2-3 sentences (`summarize_community`), largest communities first, capped at `MAX_COMMUNITIES`; `global_summary` folds the map output into one corpus-level summary.

`run_experiment` times the three stages separately so the demo can show where the runtime actually goes.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — build graph, detect communities, map/reduce summaries
# --------------------------------------------------------------------------
# Inline replacement for the repo's OllamaLLM adapter (src/llms/ollama.py):
# the same invoke / json_object contract the lab's extractor relies on.
TRIPLE_SCHEMA = '{"triples": [{"head": "...", "relation": "...", "tail": "..."}]}'


class _OllamaLLM:
    """Generate / extract with the locally served Ollama model.

    Contract mirror of the repo's OllamaLLM adapter, built inline so this
    notebook needs no repo imports: ``invoke`` returns the chat text,
    ``json_object`` strips a surrounding markdown code fence and re-prompts
    on parse failure.
    """

    def __init__(self, model: str = "qwen2.5-coder:7b", temperature: float = 0.0,
                 base_url: str = "http://localhost:11434"):
        self.model = model
        self.temperature = temperature
        self.base_url = base_url
        self._llm = None

    def _get_llm(self) -> ChatOllama:
        if self._llm is None:
            self._llm = ChatOllama(
                model=self.model, temperature=self.temperature, base_url=self.base_url
            )
        return self._llm

    def invoke(self, prompt: str) -> str:
        return self._get_llm().invoke(prompt).content

    @staticmethod
    def _strip_code_fence(text: str) -> str:
        """Remove a surrounding markdown code fence (```json ... ```)."""
        lines = text.strip().splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        return "\n".join(lines).strip()

    def json_object(self, prompt: str, retries: int = 2) -> dict:
        """Ask the model to output ONLY a JSON object and parse it."""
        text = ""
        for attempt in range(retries + 1):
            full = prompt if attempt == 0 else prompt + (
                "\n\nRespond with ONLY valid JSON, no markdown.")
            text = self.invoke(full)
            try:
                parsed = json.loads(self._strip_code_fence(text))
                if isinstance(parsed, (dict, list)):
                    return parsed
            except (json.JSONDecodeError, ValueError):
                pass
        return {"error": f"could not parse JSON after {retries + 1} attempts",
                "raw": text}


def extract_triples(llm, text: str) -> list[tuple[str, str, str]]:
    """Extract ``(head, relation, tail)`` triples from one passage of text."""
    prompt = (
        "Extract the entity-relation triples from the text below.\n"
        "Rules:\n"
        "- Only entities explicitly named in the text.\n"
        "- Entities are people, places, organizations, works, events or "
        "concrete things (1-4 words).\n"
        "- Relations are short verbs or prepositional phrases (1-4 words), "
        "present tense.\n"
        f"- Output ONLY JSON: {TRIPLE_SCHEMA}\n"
        "- Output an empty list if the text has no meaningful triples.\n"
        "\n"
        f"Text:\n{text}"
    )
    result = llm.json_object(prompt)
    if isinstance(result, list):  # bare array of triples, no wrapper key
        raw = result
    elif isinstance(result, dict) and "error" not in result:
        raw = result.get("triples", result.get("edges", result.get("data", [])))
        if isinstance(raw, dict):  # single triple given without a list wrapper
            raw = [raw]
    else:
        return []
    triples: list[tuple[str, str, str]] = []
    for item in raw or []:
        if not isinstance(item, dict):
            continue
        head = str(item.get("head", item.get("subject", ""))).strip()
        relation = str(item.get("relation", item.get("predicate", ""))).strip()
        tail = str(item.get("tail", item.get("object", ""))).strip()
        if head and tail:
            triples.append((head, relation or "related to", tail))
    return triples


def build_entity_graph(passages: list[str], llm, progress=None) -> nx.Graph:
    """Fold every passage's triples into one networkx entity graph."""
    graph = nx.Graph()
    total = len(passages)
    for i, text in enumerate(passages):
        for head, relation, tail in extract_triples(llm, text):
            if head == tail:
                continue  # self-loops carry no structure
            graph.add_edge(head, tail, relations=set())
            for node in (head, tail):
                graph.nodes[node].setdefault("passages", [])
            graph.nodes[head]["passages"].append(i)
            graph.nodes[tail]["passages"].append(i)
            graph[head][tail]["relations"].add(relation)
        if progress is not None:
            progress(i + 1, total)
    for _, _, data in graph.edges(data=True):
        data["weight"] = len(data["relations"])
    return graph


def detect_communities(graph: nx.Graph, seed: int = 42) -> list[set[str]]:
    """Partition ``graph`` into communities (Leiden, Louvain fallback).

    Isolated nodes (which Leiden drops from its membership matrix) are
    recovered as singleton communities so the partition always covers the
    whole node set.
    """
    nodes = list(graph.nodes())
    if not nodes:
        return []
    try:
        import numpy as np
        from scipy.sparse import csr_matrix
        from sknetwork.clustering import Leiden

        index = {node: i for i, node in enumerate(nodes)}
        n = len(nodes)
        rows: list[int] = []
        cols: list[int] = []
        for u, v in graph.edges():
            rows.extend((index[u], index[v]))
            cols.extend((index[v], index[u]))
        adjacency = csr_matrix(
            (np.ones(len(rows)), (rows, cols)), shape=(n, n)
        )
        membership = Leiden(random_state=seed).fit_transform(adjacency)
        dense = membership.toarray()
        labels = dense.argmax(axis=1)
        assigned = dense.sum(axis=1) > 0
    except Exception:
        # Fallback: networkx Louvain (unweighted, to match the ones/zeros above)
        parts = nx.community.louvain_communities(
            graph, weight=None, seed=seed
        )
        return [set(part) for part in parts]

    communities: dict[int, set[str]] = {}
    next_label = dense.shape[1]
    for node, label, is_assigned in zip(nodes, labels, assigned):
        if is_assigned:
            communities.setdefault(int(label), set()).add(node)
        else:
            communities[next_label] = {node}  # isolated -> own community
            next_label += 1
    return list(communities.values())


def community_text(graph: nx.Graph, community) -> str:
    """Render the intra-community edges of ``community`` as relation lines."""
    members = set(community)
    lines: set[str] = set()
    for node in members:
        for neighbor in graph.neighbors(node):
            if neighbor not in members:
                continue
            for relation in graph[node][neighbor].get("relations", ()):
                lines.add(f"{node} -[{relation}]-> {neighbor}")
    return "\n".join(sorted(lines)) or "no intra-community relations"


def summarize_community(llm, text: str) -> str:
    """Map step: condense one community's relation lines into prose."""
    prompt = (
        "Summarize the following entity-relation fragment in 2-3 sentences. "
        "Name the main entities and what connects them.\n\n"
        f"{text}"
    )
    return llm.invoke(prompt).strip()


def community_summaries(llm, graph: nx.Graph, communities, max_communities: int = 8) -> list[dict]:
    """Map/reduce-ready list of ``{"members": [...], "summary": "..."}``."""
    ordered = sorted(communities, key=len, reverse=True)
    summaries: list[dict] = []
    for community in ordered[:max_communities]:
        summaries.append(
            {
                "members": sorted(community),
                "size": len(community),
                "summary": summarize_community(
                    llm, community_text(graph, community)
                ),
            }
        )
    return summaries


def global_summary(llm, summaries) -> str:
    """Reduce step: fold all community summaries into one global summary."""
    bullets = "\n".join(f"- {summary}" for summary in summaries)
    prompt = (
        "Combine the following community summaries into one global summary "
        "of the corpus in 3-4 sentences.\n\n"
        f"{bullets}"
    )
    return llm.invoke(prompt).strip()


def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    llm = _OllamaLLM()  # local qwen2.5-coder:7b; map/reduce + extraction

    t0 = time.perf_counter()
    graph = build_entity_graph(passages, llm)
    build_s = time.perf_counter() - t0

    communities = detect_communities(graph, seed=SEED)
    t0 = time.perf_counter()
    summaries = community_summaries(
        llm, graph, communities, max_communities=MAX_COMMUNITIES
    )
    map_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    global_text = global_summary(llm, [s["summary"] for s in summaries])
    reduce_s = time.perf_counter() - t0

    covered = {node for community in communities for node in community}
    return {
        "graph": graph,
        "communities": communities,
        "summaries": summaries,
        "global_summary": global_text,
        "coverage": len(covered),
        "build_s": build_s,
        "map_s": map_s,
        "reduce_s": reduce_s,
    }


## 4. Demo — print the artifact

The demo prints the artifact the index will later be queried against: entity/community counts and coverage, the community size distribution, the map summaries, the reduced global summary, and per-stage timing. Coverage is the property that makes the index sound — every entity belongs to exactly one community, so no part of the corpus is invisible to a later local/global query.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-03 — Communities + map/reduce summarization")
    graph = exp["graph"]
    print(f"{graph.number_of_nodes()} entities, {len(exp['communities'])} "
          f"communities, coverage {exp['coverage']}/{graph.number_of_nodes()}")
    print("=" * 66)

    sizes = sorted((len(c) for c in exp["communities"]), reverse=True)
    print(f"\n[1] Community sizes (largest first): {sizes}")

    print(f"\n[2] Map — community summaries (top {MAX_COMMUNITIES} by size):")
    for i, entry in enumerate(exp["summaries"], start=1):
        print(f"    C{i} (size {entry['size']}): {entry['summary']}")

    print(f"\n[3] Reduce — global corpus summary:")
    print(f"    {exp['global_summary']}")

    print(f"\n[4] Timing: build graph {exp['build_s']:.1f}s, "
          f"map {exp['map_s']:.1f}s, reduce {exp['reduce_s']:.1f}s")

    print(f"\n[5] Takeaway")
    print("    Communities partition the corpus by structure, not by query.")
    print("    The map step compresses every community into a few sentences;")
    print("    the reduce step folds them into one global summary. Lab 04")
    print("    queries this index two ways: local (walk the graph from the")
    print("    question's entities) and global (rank the summaries).")


## 5. Verification gate

The lab ships a `--verify` gate — the same gate `python src/curriculum/08-graphrag/03-communities.py --verify` runs: hard checks the index must clear — every entity covered by a community, communities disjoint, at least two communities, the summary cap respected, and no empty summaries (neither per-community nor the global one). This turns "the lab ran" into "the lab ran *correctly*": the index is structurally sound before Lab 04 ever queries it.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    graph = exp["graph"]
    communities = exp["communities"]
    covered = exp["coverage"]

    checks.append(("communities cover every entity (no orphan nodes)",
                   covered == graph.number_of_nodes()))
    checks.append(("communities are disjoint",
                   sum(len(c) for c in communities) == covered))
    checks.append((f"at least 2 communities (got {len(communities)})",
                   len(communities) >= 2))
    checks.append((f"summaries cover {len(exp['summaries'])} communities "
                   f"(cap {MAX_COMMUNITIES})",
                   len(exp["summaries"]) <= MAX_COMMUNITIES))
    checks.append(("every summary is non-empty",
                   all(entry["summary"].strip()
                       for entry in exp["summaries"])))
    checks.append(("global summary is non-empty",
                   bool(exp["global_summary"].strip())))
    checks.append(("community sizes are consistent with members",
                   all(len(entry["members"]) == entry["size"]
                       for entry in exp["summaries"])))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

This is the slow cell: ~20 LLM extraction calls to build the graph, up to 6 community summaries, and 1 reduce call — a few minutes against a local `qwen2.5-coder:7b`. No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The index Lab 04 will query: community sizes, the per-community map summaries, the reduced global summary, and the per-stage timing.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b` and that the corpus parquet is intact.


In [ ]:
verify_gate(exp)
